# Generating embeddings using TorchGeo

## 1. Project overview 

The goal of this project is to develop a robust representation learning pipeline for agricultural monitoring. Instead of building a simple "black-box" classifier, we are focusing on **embedding generation**, transforming complex satellite time-series data into a low-dimensional, meaningful vector space.

**Key Objectives:**
* **Time-Series Embeddings:** Use `TorchGeo` enconders to compress temporal and spectral data into a 1D feature vector.
* **Binary Classification Focus:** To maximize data density and global applicability, we are focusing on the binary classification (crop vs. non-crop) using the *GeoWiki-landcover-2017* sub-dataset.
* **Feature Extraction:** Leveraging the latent space of a model to distinguish agricultural patterns from natural land covers and analyse if different geographical locations produce similar embeddings for the same land class.


## 2. Dataset organization

### Downloading the CropHarvest dataset

In [ ]:
from torchgeo.datasets import CropHarvest

root = "_data"
original_dataset = CropHarvest(root=root, download=True)

In [ ]:
# pandas dataframe containing label data for everything on CropHarvest
original_dataset._load_labels(root).head(5)

### Exploring the dataset

In [ ]:
# under properties.dataset we can find the names of the "sub datasets" inside CropHarvest
original_dataset_counts = original_dataset.labels['properties.dataset'].value_counts()

#  all the subdataset inside the CropHarvest dataset and the number of its samples
print("Sub-datasets inside CropHarvest:")
print(original_dataset_counts)

In [ ]:
# group by the sub dataset name and look at the 'properties.label' column, which contains the crop type names
# if it is binary it will be None
# if it is multiclass it will have the crop type names

original_dataset_class_search = original_dataset.labels.groupby('properties.dataset')['properties.label'].unique()

binary = []
multiclass = []

for sub_dataset, labels in original_dataset_class_search.items():
    labels = [l for l in labels if l is not None and not (isinstance(l, float))]

    if len(labels) > 0:
        multiclass.append((sub_dataset, labels))
    else:
        binary.append(sub_dataset)

print("Binary sub-datasets:")
for i in binary:
    print(i)

print("\nMulticlass sub-datasets:")
for j in multiclass:
    print(j)

### Isolate *geowiki-landcover-2017* sub dataset


We want to work with an isolated sub-set, binary, and preferably with lots of samples. For that reason it is wise to choose *geowiki-landcover-2017*
- **Sub dataset name:** *geowiki-landcover-2017*
- **Sample Count:** 35866 points
- **Format:** pixel-level time series

In [ ]:
from torch.utils.data import Subset

# finding all the entries on original_dataset that has properties.dataset equal to 'geowiki-landcover-2017'
DATASET_NAME = 'geowiki-landcover-2017'
geowiki_indices = original_dataset.labels.index[original_dataset.labels['properties.dataset'] == DATASET_NAME].tolist()

# create the isolated dataset
geowiki_dataset = Subset(original_dataset, geowiki_indices)

print(f"Original dataset size: {len(original_dataset)}")
print(f"Isolated dataset size: {len(geowiki_dataset)}")

In [ ]:
geowiki_dataset.__getitem__(35865)

### Turning *geowiki-landcover-2017* into a binary dataset

But we can see that, even though the sub dataset *geowiki-landcover-2017* is binary and has no `properties.label` associated with, there are many possible values for the label, meaning that has in fact is a multi classes.

For that reason we'll apply a transformation on the label tensor, making sure that all 0 label values remain 0 (non crop) and that any other value is turned into 1 (crop).

In [ ]:
import torch

class BinaryDatasetWrapper(torch.utils.data.Dataset):
    def __init__(self, subset):
        self.subset = subset

    def __getitem__(self, index):
        sample = self.subset[index]
        
        sample['label'] = torch.tensor(1) if sample['label'] > 0 else torch.tensor(0)
        return sample

    def __len__(self):
        return len(self.subset)

dataset = BinaryDatasetWrapper(geowiki_dataset)

In [ ]:
print("Original geowiki dataset:")
print(geowiki_dataset[45]['label'])
print(geowiki_dataset[35865]['label'])
print(geowiki_dataset[0]['label'])

print("\nBinary geowiki dataset:")
print(dataset[45]['label'])
print(dataset[35865]['label'])
print(dataset[0]['label'])

## 3. *geowiki-landcover-2017* data exploration

### Data structure

Each data point represents a single geographic location over a full calendar year. Each sample is provided as a $12 \times 18$ tensor:
- Rows (12): Represent the **12 monthsof the year (temporal dimension)
- Columns (18): Represent the spectral and environmental bands (feature dimension)

The data combines multiple sensors and environmental variables:

| Band Index | Feature Category | Description |
| :--- | :--- | :--- |
| **0 - 12** | **Sentinel-2 (Optical)** | Multi-spectral bands including RGB, NIR, and Red-Edge |
| **13 - 14** | **Sentinel-1 (Radar)** | VV and VH polarizations (crucial for cloud-penetrating growth monitoring) |
| **15** | **ERA5 (Temp)** | 2-meter temperature data (average monthly) |
| **16** | **ERA5 (Precip)** | Total monthly precipitation |
| **17** | **SRTM (Elevation)** | Digital Elevation Model (Static value repeated across time steps) |

In [ ]:
dataset[0].keys()

In [ ]:
sample = dataset.__getitem__(0)
sample

In [ ]:
sample['array'].shape # data itself

In [ ]:
sample['label'].shape # binary label

### Data visualization

In [ ]:
import matplotlib.pyplot as plt

# the actual tensor
plt.imshow(sample['array'], origin='lower')
plt.title("12 x 18 tensor")
plt.show()

In [ ]:
# it shows two plots because it is prepared to plot ground-truth and 
# it combines the rgb channels into a single pixel 
original_dataset.plot(sample)

In [ ]:
# # load the crop-type label for a single pixel time series
# original_dataset._load_label(45, dataset='geowiki-landcover-2017')

# para ver depois porque é que isto acontece

In [ ]:
def get_class_balance(subset_obj):
    if not hasattr(subset_obj, 'indices'):
        indices = range(len(subset_obj))
    else:
        indices = subset_obj.indices

    curr = subset_obj
    while not hasattr(curr, 'labels'):
        if hasattr(curr, 'dataset'):
            curr = curr.dataset
        elif hasattr(curr, 'subset'):
            curr = curr.subset
        else:
            break
    
    if hasattr(curr, 'labels'):
        labels = curr.labels.iloc[indices]['properties.is_crop']
        counts = labels.value_counts().to_dict()
        
        crop_count = counts.get(1, 0)
        non_crop_count = counts.get(0, 0)
        total = len(indices)
        
        print(f"Total Samples: {total}")
        print(f"  - Crop (1):     {crop_count} ({100*crop_count/total:.1f}%)")
        print(f"  - Non-Crop (0): {non_crop_count} ({100*non_crop_count/total:.1f}%)")
        return counts
    else:
        print("Error: Could not find labels in the dataset hierarchy.")
        return None

In [ ]:
get_class_balance(dataset)

In [ ]:
plt.figure(figsize=(11, 6))

geowiki_metadata = original_dataset.labels.iloc[geowiki_indices]

for is_crop, color, label in [(0, 'red', 'Non-Crop'), (1, 'blue', 'Crop')]:
    mask = geowiki_metadata['properties.is_crop'] == is_crop
    subset = geowiki_metadata[mask]
    plt.scatter(subset['properties.lon'], subset['properties.lat'], s=1, alpha=0.15, c=color, label=label)

plt.legend(markerscale=10)
plt.title("spatial distribution of crop and non crop classes")
plt.show()

## 4. Creating the embeddings model

### Train/test split

In [ ]:
from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

print("Train set size:", len(train_dataset))
print("Test set size:", len(test_dataset))

In [ ]:
# class balance in train set
get_class_balance(train_dataset)

In [ ]:
# class balance in test set
get_class_balance(test_dataset)

### DOFA model

Instead of trying to retrain the whole DOFA model (which is huge and requires a lot of GPU memory), we will use DOFA as a "frozen" backbone to generate embeddings and train a simple, fast classifier on top.

In [ ]:
import lightning as L
from torch.utils.data import DataLoader

class GeoWikiDataModule(L.LightningDataModule):
    def __init__(self, train_ds, test_ds, batch_size: int = 32, num_workers: int = 0):
        super().__init__()
        self.train_ds = train_ds
        self.test_ds = test_ds
        self.batch_size = batch_size
        self.num_workers = num_workers

    def train_dataloader(self):
        return DataLoader(
            self.train_ds, 
            batch_size=self.batch_size, 
            shuffle=True, 
            num_workers=self.num_workers
        )

    def val_dataloader(self):
        return DataLoader(
            self.test_ds, 
            batch_size=self.batch_size, 
            num_workers=self.num_workers
        )

    def test_dataloader(self):
        return DataLoader(
            self.test_ds, 
            batch_size=self.batch_size, 
            num_workers=self.num_workers
        )

In [ ]:
import torch
from torch.utils.data import DataLoader

# 1. Define the 'device' (Checks for Mac M1/M2, Nvidia GPU, or falls back to CPU)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps") # For Mac users
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

# # 2. Create the DataLoaders from YOUR train_dataset and test_dataset
# # These are the variables you created with random_split earlier
# train_loader = DataLoader(train_dataset, batch_size=64, shuffle=False)
# test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

try this after

In [ ]:
# istead of the code above i could have used this one
# Instead of manual DataLoaders, you can use your DataModule:


dm = GeoWikiDataModule(train_dataset, test_dataset, batch_size=64)
train_loader = dm.train_dataloader()
test_loader = dm.test_dataloader()

In [ ]:
from torchgeo.models import DOFABase16_Weights, get_model
import kornia.augmentation as K

model = get_model('dofa_base_patch16_224', weights=DOFABase16_Weights.DOFA_MAE)
model = model.eval()#.to(accelerator)

# augs = K.AugmentationSequential(
#     K.Normalize(
#         mean=0.0, std=10_000, p=1.0
#     ),  # Divide by 10,000 to approximately scale to [0, 1]
#     K.Resize((224, 224), antialias=True),  # DOFA model expects 224x224 inputs
# )

In [ ]:
# import torch
# import torch.nn.functional as F
# import numpy as np
# from tqdm import tqdm

# def embed_dofa(model, dataloader, device):
#     """
#     Updated helper function for CropHarvest time-series (12 months x 18 bands).
#     """
#     model.eval()
#     x_all = []
#     y_all = []
    
#     # The 18 wavelengths for CropHarvest
#     wavelengths = [
#         0.0, 0.0, # S1
#         490.0, 560.0, 665.0, 842.0, 1610.0, 2190.0, 865.0, 705.0, 740.0, 783.0, # S2
#         0.0, 0.0, 0.0, 0.0, 0.0, 0.0 # Others
#     ]

#     for batch in tqdm(dataloader):
#         # 1. Access the correct key 'array'
#         # x = batch['array'].to(device) # Shape: (B, 12, 18)
#         x = batch['array'].to(torch.float32).to(device)
#         y = batch['label']

#         # # 2. Reformat and Pad for DOFA
#         # # Transform (B, 12, 18) -> (B, 18, 12)
#         # x = x.transpose(1, 2).float()
#         # # Pad months from 12 to 16 and add dummy width
#         # x = F.pad(x, (0, 4)).unsqueeze(-1) # Shape: (B, 18, 16, 1)

#         # --- Replace your current padding logic with this ---
#         # 1. Transform (B, 12, 18) -> (B, 18, 12)
#         x = x.transpose(1, 2)

#         # 2. Pad time (12 -> 16) AND pad dummy width (1 -> 16)
#         # F.pad(x, (pad_left, pad_right, pad_top, pad_bottom))
#         # We pad the last dimension (width) to 16 and the second to last (time) to 16
#         x = F.pad(x, (0, 15, 0, 4)) # Now shape is (B, 18, 16, 16)

#         with torch.inference_mode():
#             # 3. Extract features using the full 18-wavelength list
#             # We use the backbone/forward_features to get the 768-dim vector
#             # Note: Using the CLS token (index 0)
#             feats = model(x, wavelengths=wavelengths)
#             # feats = model.backbone(x, wavelengths=wavelengths)
            
#             if feats.ndim == 3:
#                 embeddings = feats[:, 0, :] # Extract CLS token
#             else:
#                 embeddings = feats

#         x_all.append(embeddings.cpu().numpy())
#         y_all.append(y.numpy())

#     # Concatenate all batches
#     x_all = np.concatenate(x_all, axis=0)
#     y_all = np.concatenate(y_all, axis=0)
    
#     return x_all, y_all

In [ ]:
# train_dl = datamodule.train_dataloader()
# val_dl = datamodule.val_dataloader()

In [ ]:
# def embed_dofa(model, dataloader, device):
#     model.eval()
#     model.to(device)
#     x_all, y_all = [], []
    
#     wavelengths = [0.0, 0.0, 490.0, 560.0, 665.0, 842.0, 1610.0, 2190.0, 
#                    865.0, 705.0, 740.0, 783.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

#     for batch in tqdm(dataloader):
#         # 1. Load: (Batch, Time=12, Bands=18)
#         x = batch['array'].to(torch.float32).to(device) 
#         y = batch['label']

#         # 2. Reorder: (Batch, Bands=18, Time=12)
#         x = x.permute(0, 2, 1) 

#         # 3. Add a dummy width dimension: (Batch, 18, 12, 1)
#         # This makes it a 4D tensor (N, C, H, W)
#         x = x.unsqueeze(-1)

#         # 4. Pad: H (12 -> 16) and W (1 -> 16)
#         # For 4D input, pad uses (left, right, top, bottom)
#         x = F.pad(x, (0, 15, 0, 4)) # Result: (Batch, 18, 16, 16)

#         # with torch.inference_mode():
#         #     curr_model = model.backbone if hasattr(model, 'backbone') else model
            
#         #     # Now x is guaranteed (Batch, 18, 16, 16)
#         #     feats = curr_model(x, wavelengths=wavelengths)
            
#         #     # DOFA returns [Batch, Tokens, 768]. CLS token is index 0.
#         #     embeddings = feats[:, 0, :]

#         # x_all.append(embeddings.cpu().numpy())
#         with torch.inference_mode():
#             curr_model = model.backbone #if hasattr(model, 'backbone') else model
            
#             # The model returns a pooled 2D tensor [Batch, 768]
#             feats = curr_model(x, wavelengths=wavelengths)
            
#             # FIX: Check the shape before slicing
#             if feats.ndim == 3:
#                 # If it's 3D, we take the first token (CLS)
#                 embeddings = feats[:, 0, :]
#             else:
#                 # If it's already 2D, the model already pooled it for us!
#                 embeddings = feats

#         x_all.append(embeddings.cpu().numpy())
#         y_all.append(y.numpy())


        

#     return np.concatenate(x_all, axis=0), np.concatenate(y_all, axis=0)

down here is the good versiom

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm

def embed_dofa(model, dataloader, device):
    model.eval()
    model.to(device)
    x_all, y_all = [] , []
    
    # 1. Use ALL 18 wavelengths (in nm) for CropHarvest
    # S1 (2), S2 (10), Weather/Elevation (6)
    wavelengths = [60.0, 60.0, 490.0, 560.0, 665.0, 842.0, 1610.0, 2190.0, 
                   865.0, 705.0, 740.0, 783.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

    for batch in tqdm(dataloader):
        # Use 'array' key for CropHarvest; convert to float32 for MPS compatibility
        x = batch['array'].to(torch.float32).to(device) 
        y = batch['label']

        # 2. Geometry Fix for Time-Series
        # (Batch, 12, 18) -> (Batch, 18, 12)
        x = x.permute(0, 2, 1) 
        # Add dummy width and pad to 16x16 so the 2D kernel fits
        x = x.unsqueeze(-1)
        x = F.pad(x, (0, 15, 0, 4)) # Result: (Batch, 18, 16, 16)

        with torch.inference_mode():
            # 3. Use forward_features as per your snippet to get the raw 768-dim vector
            # This bypasses any task-specific heads
            feats = model.forward_features(x, wavelengths=wavelengths)
            
            # 4. Handle pooling (if model returns [B, Tokens, 768] vs [B, 768])
            if feats.ndim == 3:
                embeddings = feats.mean(dim=1) # Global Average Pooling
            else:
                embeddings = feats

        x_all.append(embeddings.cpu().numpy())
        y_all.append(y.numpy())

    return np.concatenate(x_all, axis=0), np.concatenate(y_all, axis=0)

In [ ]:
model = model.to(device) # Force everything to MPS

print("Extracting training embeddings...")
x_train, y_train = embed_dofa(model, train_loader, device)

print("Extracting test embeddings...")
x_test, y_test = embed_dofa(model, test_loader, device)

print(f"Corrected! Created {x_train.shape[0]} vectors of size {x_train.shape[1]}")

In [ ]:
import numpy as np

output_folder = "_results/"

# Save as binary files
np.save(f'{output_folder}x_train_dofa3.npy', x_train)
np.save(f'{output_folder}y_train_dofa3.npy', y_train)
np.save(f'{output_folder}x_test_dofa3.npy', x_test)
np.save(f'{output_folder}y_test_dofa3.npy', y_test)

print("Files saved successfully!")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. Initialize and Train
rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
rf.fit(x_train, y_train)

# 2. Predict
y_pred = rf.predict(x_test)

# 3. Evaluate
print(f"Random Forest Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns

# We'll use a subset of 1000 points to make it fast
tsne = TSNE(n_components=2, random_state=42)
x_subset = x_test[:1000]
y_subset = y_test[:1000]
x_2d = tsne.fit_transform(x_subset)

# Plot
plt.figure(figsize=(10, 7))
sns.scatterplot(x=x_2d[:,0], y=x_2d[:,1], hue=y_subset, palette='viridis', alpha=0.7)
plt.title("DOFA Embedding Space (Crop vs Non-Crop)")
plt.show()